# 07b — Factorized Work-Regime Profiles

This stage preserves overlapping mobility evidence axes instead of assigning one mutually-exclusive work class.

It does **not** infer occupation, employment status, true workplace, OFFICE, or semantic POI ground truth.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/tanh1c/geolife.git'
BRANCH = 'eda/07b-factorized-work-profiles'
REPO_DIR = Path('/root/geolife')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'], check=True)

print('repo:', REPO_DIR)
print('sha:', subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'], check=True, capture_output=True, text=True).stdout.strip())


In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
import pandas as pd

def load_module(name, path):
    spec = spec_from_file_location(name, path)
    assert spec is not None and spec.loader is not None
    module = module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

stage07b = load_module('stage07b', REPO_DIR / 'analysis' / '07b_factorized_work_profiles.py')
print(stage07b.synthetic_self_check())


In [ ]:
VOLUME_ROOT = Path('/mnt/geolife-data')
CACHE_07B = VOLUME_ROOT / 'cache' / '07b_factorized_work_profiles'
CACHE_07B.mkdir(parents=True, exist_ok=True)

def find_required(filename, preferred=()):
    for path in preferred:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob(f'**/{filename}'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Cannot find required artifact: {filename}')

def find_optional(filename, preferred=()):
    for path in preferred:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob(f'**/{filename}'))
    return matches[0] if matches else None

FEATURES_PATH = find_required('user_behavior_features.csv', [
    VOLUME_ROOT / 'cache' / '03a_user_behavior_deep_dive' / 'user_behavior_features.csv',
])
HOME_PATH = find_required('home_consensus_private.pkl', [
    VOLUME_ROOT / 'cache' / '05b_home_consensus_adaptive_work' / 'home_consensus_private.pkl',
])
WORK_PATH = find_required('adaptive_work_patterns_42_private.pkl', [
    VOLUME_ROOT / 'cache' / '05b_home_consensus_adaptive_work' / 'adaptive_work_patterns_42_private.pkl',
])
ROUTINE_PATH = find_required('user_summary_private.pkl', [
    VOLUME_ROOT / 'cache' / '06_routine_habit_mining' / 'user_summary_private.pkl',
])
INDEPENDENT_PATH = find_optional('candidate_comparison_private.pkl', [
    VOLUME_ROOT / 'cache' / '05c_stable_secondary_evidence' / 'candidate_comparison_private.pkl',
])

print('03a:', FEATURES_PATH)
print('05b home:', HOME_PATH)
print('05b work:', WORK_PATH)
print('06:', ROUTINE_PATH)
print('05c optional:', INDEPENDENT_PATH)


In [ ]:
behavior_features = pd.read_csv(FEATURES_PATH, dtype={'user_id': str})
home_evidence = pd.read_pickle(HOME_PATH)
work_patterns = pd.read_pickle(WORK_PATH)
routine_users = pd.read_pickle(ROUTINE_PATH)
independent = pd.read_pickle(INDEPENDENT_PATH) if INDEPENDENT_PATH else None

needed_flags = {'stable_shifted_candidate', 'mobile_work_like_candidate'}
if not needed_flags.issubset(behavior_features.columns):
    try:
        import geolife  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=True)
    stage03a = load_module('stage03a_for_07b', REPO_DIR / 'analysis' / '03a_user_behavior_deep_dive.py')
    behavior_features = stage03a.assign_behavioral_candidates(behavior_features)

print('behavior users:', behavior_features['user_id'].astype(str).nunique())
print('work-pattern users:', work_patterns['user_id'].astype(str).nunique())
print('routine users:', routine_users['user_id'].astype(str).nunique())
print('independent-evidence users:', 0 if independent is None else independent['user_id'].astype(str).nunique())


In [ ]:
audit = stage07b.run_audit(
    behavior_features,
    work_patterns,
    routine_users,
    home_evidence=home_evidence,
    independent_secondary_evidence=independent,
)

print('SUPPORT SUMMARY')
display(audit.support_summary)

print('AXIS SUMMARY')
display(audit.axis_summary.sort_values('users_true', ascending=False))

print('PAIRWISE OVERLAP — largest intersections')
display(audit.pairwise_overlap.sort_values(['intersection_users','jaccard'], ascending=[False,False]).head(30))

print('EVIDENCE SIGNATURES')
display(audit.signature_summary.head(30))

print('REPRESENTATION SIGNATURES')
display(audit.representation_summary)


In [ ]:
profiles = audit.profiles

print('OVERLAP CHECKS FROM STAGE 07 V1')
checks = pd.DataFrame([
    {
        'check': 'stable_secondary AND multiple_recurring',
        'users': int((profiles['site_stable_secondary'] & profiles['site_multiple_recurring']).sum()),
    },
    {
        'check': 'stable_secondary AND repeated_route',
        'users': int((profiles['site_stable_secondary'] & profiles['route_repeated']).sum()),
    },
    {
        'check': 'multiple_recurring AND repeated_route',
        'users': int((profiles['site_multiple_recurring'] & profiles['route_repeated']).sum()),
    },
    {
        'check': 'shifted AND repeated_route',
        'users': int((profiles['schedule_shifted_evidence'] & profiles['route_repeated']).sum()),
    },
])
display(checks)


In [ ]:
# User-level profiles remain private on the Modal Volume.
audit.profiles.to_pickle(CACHE_07B / 'factorized_work_profiles_private.pkl')

for name, frame in {
    'support_summary': audit.support_summary,
    'axis_summary': audit.axis_summary,
    'pairwise_overlap': audit.pairwise_overlap,
    'signature_summary': audit.signature_summary,
    'representation_summary': audit.representation_summary,
}.items():
    frame.to_csv(CACHE_07B / f'{name}.csv', index=False)

print('saved:', CACHE_07B)


## Interpretation boundary

- Overlap between axes is expected and is the main object of Stage 07b.
- A representation option is geometry, not semantic WORK truth.
- No axis or signature is an occupation label.
- External POI / land-use context, if added next, must remain an independent semantic layer.
